In [ ]:
# ===============================
# 1 Import Libraries
# ===============================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import flwr as fl  

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)

In [ ]:
# ===============================
# 2 GPU Configuration
# ===============================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs:", gpus)

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict


def get_patient_id(filename):
    """
    Extract patient ID from filename like:
    OAS1_0001_MR1_mpr-1_100.jpg -> 0001
    """
    return filename.split("_")[1]


def split_dataset_patientwise(
    input_dir, output_dir, split_ratio=(0.7, 0.15, 0.15), seed=42
):
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    train_ratio, val_ratio, test_ratio = split_ratio
    assert round(train_ratio + val_ratio + test_ratio, 5) == 1.0

    classes = ["Mild Dementia", "Non Dementia", "Very mild Dementia"]
    splits = ["train", "val", "test"]

    # Create folder structure
    for split in splits:
        for cls in classes:
            (output_path / split / cls).mkdir(parents=True, exist_ok=True)

    random.seed(seed)

    for cls in classes:
        cls_dir = input_path / cls

        if not cls_dir.exists():
            print(f"Warning: {cls_dir} not found")
            continue

        #  Group files by patient ID
        patient_dict = defaultdict(list)

        for f in cls_dir.iterdir():
            if f.is_file() and not f.name.startswith("."):
                pid = get_patient_id(f.name)
                patient_dict[pid].append(f)

        patients = list(patient_dict.keys())
        random.shuffle(patients)

        total_patients = len(patients)
        train_idx = int(total_patients * train_ratio)
        val_idx = train_idx + int(total_patients * val_ratio)

        train_patients = patients[:train_idx]
        val_patients = patients[train_idx:val_idx]
        test_patients = patients[val_idx:]

        def copy_patient_files(patient_list, split_name):
            count = 0
            for pid in patient_list:
                for f in patient_dict[pid]:
                    dest = output_path / split_name / cls / f.name
                    shutil.copy2(f, dest)
                    count += 1
            return count

        train_count = copy_patient_files(train_patients, "train")
        val_count = copy_patient_files(val_patients, "val")
        test_count = copy_patient_files(test_patients, "test")

        print(f"\nClass: {cls}")
        print(
            f"Patients -> Train: {len(train_patients)}, Val: {len(val_patients)}, Test: {len(test_patients)}"
        )
        print(f"Images   -> Train: {train_count}, Val: {val_count}, Test: {test_count}")


# --- Run ---
split_dataset_patientwise(
    input_dir="Data", output_dir="Dataset_split", split_ratio=(0.70, 0.15, 0.15)
)

In [ ]:
TRAIN_PATH = r"Dataset_split/train"
VAL_PATH = r"Dataset_split/val"
TEST_PATH = r"Dataset_split/test"

In [ ]:
#new
import os
import shutil
from collections import defaultdict

BASE_PATH = "Dataset_split/train"
OUTPUT_PATH = "federated_clients"
NUM_CLIENTS = 5

CLASSES = ["Mild Dementia", "Non Dementia", "Very mild Dementia"]

# Step 1: group patients
class_patients = defaultdict(list)

for cls in CLASSES:
    class_path = os.path.join(BASE_PATH, cls)
    for file in os.listdir(class_path):
        if file.endswith(".jpg"):
            pid = file.split("_")[1]
            if pid not in class_patients[cls]:
                class_patients[cls].append(pid)

# Step 2: split patients per class evenly
clients = {i: [] for i in range(NUM_CLIENTS)}

for cls in CLASSES:
    patients = class_patients[cls]

    chunk_size = len(patients) // NUM_CLIENTS

    for i in range(NUM_CLIENTS):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i != NUM_CLIENTS - 1 else len(patients)

        for pid in patients[start:end]:
            clients[i].append((pid, cls))

# Step 3: create folders
for i in range(NUM_CLIENTS):
    for cls in CLASSES:
        os.makedirs(os.path.join(OUTPUT_PATH, f"client_{i}", cls), exist_ok=True)

# Step 4: copy files
for client_id, patient_list in clients.items():
    for pid, cls in patient_list:
        class_path = os.path.join(BASE_PATH, cls)

        for file in os.listdir(class_path):
            if file.endswith(".jpg") and file.split("_")[1] == pid:
                src = os.path.join(class_path, file)
                dst = os.path.join(OUTPUT_PATH, f"client_{client_id}", cls, file)

                shutil.copy(src, dst)

print("✅ Balanced + patient-wise split DONE!")

In [2]:
import flwr as fl
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.applications.densenet import preprocess_input

In [3]:
# ===============================
# CONFIG
# ===============================
TRAIN_DIR = "federated_clients/client_2"
VAL_DIR = "Dataset_split/val"
TEST_DIR = "Dataset_split/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 1
ROUNDS = 12

# Weights to address class imbalance
class_weights = {0: 3.75, 1: 0.5, 2: 1.5}
CLASSES = ["Mild Dementia", "Non Dementia", "Very mild Dementia"]

In [4]:
# ===============================
# DATA LOADING
# ===============================


# LIGHT augmentation (safe for MRI)
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.05),
    ]
)


# ===============================
# TRAIN DATA (CLIENT)
# ===============================
def load_client_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
    )

    # Normalize
    ds = ds.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    #  Apply augmentation
    ds = ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    return ds.prefetch(tf.data.AUTOTUNE)


# ===============================
# VALIDATION DATA
# ===============================
def load_val_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        label_mode="categorical",
    )

    ds = ds.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    return ds.cache().prefetch(tf.data.AUTOTUNE)


# ===============================
# TEST DATA
# ===============================
def load_test_data():
    ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        label_mode="categorical",
    )

    ds = ds.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    return ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
# ===============================
# MODEL
# ===============================
import tensorflow as tf
from tensorflow.keras import layers, models


#  Stable focal loss
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)

        return tf.reduce_sum(weight * cross_entropy, axis=-1)

    return loss


def create_model():
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights="imagenet", input_shape=(224, 224, 3)
    )

    #  Freeze more layers (important for FL)
    for layer in base.layers[:-80]:
        layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.6)(x)

    output = layers.Dense(len(CLASSES), activation="softmax")(x)

    model = models.Model(inputs=base.input, outputs=output)

    #  VERY IMPORTANT (stable FL)
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-5, clipnorm=1.0  
    )

    model.compile(
        optimizer=optimizer,
        loss=focal_loss(gamma=2.0, alpha=0.25),
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
    )

    return model

In [5]:
class FlowerClient(fl.client.NumPyClient):
    def __init__(self):
        self.model = create_model()
        self.train_data = load_client_data()
        self.val_data = load_val_data()

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        current_round = config.get("server_round", 0)
        print(f"Training Client - Round {current_round}")

        history = self.model.fit(
            self.train_data, epochs=EPOCHS, class_weight=class_weights, verbose=1
        )

        #  FIXED sample count
        num_examples = sum([x.shape[0] for x, _ in self.train_data])

        results = {
            "accuracy": float(history.history["accuracy"][-1]),
            "recall": float(history.history.get("recall", [0])[-1]),
        }

        #  safer saving condition
        if current_round == ROUNDS:
            self.model.save("final_alzheimer_model.h5")
            print(f" Final round {current_round} complete. Model saved.")

        return self.model.get_weights(), num_examples, results

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)

        loss, acc, recall = self.model.evaluate(self.val_data, verbose=0)

        #  FIXED sample count
        num_examples = sum([x.shape[0] for x, _ in self.val_data])

        return loss, num_examples, {"accuracy": acc, "recall": recall}

In [ ]:
# ===============================
# START CLIENT
# ===============================
fl.client.start_numpy_client(server_address="your ip address:8080", client=FlowerClient())

NameError: name 'fl' is not defined

In [ ]:


# Load the best model saved during the federated rounds
# Replace 'best_alzheimer_model.h5' with your actual filename
model = tf.keras.models.load_model(
    "final_alzheimer_model.h5", custom_objects={"loss": focal_loss()}
)

In [ ]:
y_true = []
y_pred = []
test_data = load_test_data()
for x, y in test_data:
    preds = model.predict(x, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(y.numpy(), axis=1))